In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, DoubleType, IntegerType
from pyspark.sql.window import Window

bronze = spark.table("bank_pulse.bronze_financials")

# Select and rename in one step — avoids the drop/case-sensitivity issue
silver = bronze.select(
    F.to_date(F.col("REPDTE"), "yyyyMMdd").alias("report_date"),
    F.col("CERT").cast(IntegerType()).alias("cert"),
    F.col("ASSET").cast(LongType()).alias("total_assets"),
    F.col("DEP").cast(LongType()).alias("total_deposits"),
    F.col("LNLSNET").cast(LongType()).alias("net_loans"),
    F.col("INTINC").cast(DoubleType()).alias("interest_income"),
    F.col("EINTEXP").cast(DoubleType()).alias("interest_expense"),
    F.col("NETINC").cast(DoubleType()).alias("net_income"),
    F.col("NPERFV").cast(DoubleType()).alias("nonperforming_assets"),
    F.col("RBC1RWAJ").cast(DoubleType()).alias("tier1_capital_ratio"),
    F.col("ROA").cast(DoubleType()).alias("roa"),
    F.col("ROE").cast(DoubleType()).alias("roe"),
    F.col("ingested_at"),
    F.col("source"),
    F.col("batch_id")
)

# Deduplicate
dedup_window = Window.partitionBy("cert", "report_date").orderBy(F.col("ingested_at").desc())
silver = (silver
    .withColumn("_rank", F.row_number().over(dedup_window))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
)

# Data quality flags
silver = (silver
    .withColumn("dq_missing_assets",        F.col("total_assets").isNull())
    .withColumn("dq_negative_assets",       F.col("total_assets") < 0)
    .withColumn("dq_missing_capital_ratio", F.col("tier1_capital_ratio").isNull())
    .withColumn("dq_missing_report_date",   F.col("report_date").isNull())
)

# Write
(silver.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("report_date")
    .saveAsTable("bank_pulse.silver_financials")
)

print("✓ Silver written")

display(spark.sql("""
    SELECT
        COUNT(*)                                   AS total_rows,
        SUM(CAST(dq_missing_assets AS INT))        AS missing_assets,
        SUM(CAST(dq_negative_assets AS INT))       AS negative_assets,
        SUM(CAST(dq_missing_capital_ratio AS INT)) AS missing_capital_ratio,
        SUM(CAST(dq_missing_report_date AS INT))   AS missing_report_date
    FROM bank_pulse.silver_financials
"""))